# Qwen3-Embedding Retrieval Fine-Tuning on DBpedia (MNRL)

**Purpose:** Fine-tune `Qwen/Qwen3-Embedding-0.6B` on SPARQL semantic triplets using a **retrieval-aligned** objective that mirrors how `SPARQLCE` is trained in `01_train_dbpedia.ipynb`.

| qwen_finetuning | This notebook |
|---|---|
| Loss | `MultipleNegativesRankingLoss` (in-batch InfoNCE) |
| Negatives per anchor | 1 hard + 2·(B-1) in-batch |
| Data  | `SPARQL-Embeddings/dataset/raw/jaccard_triplets_dbpedia.json` + `splits/splits_dbpedia.json` (same data and split as SPARQLCE) |
| Evaluator  | `InformationRetrievalEvaluator` (MRR/nDCG/Recall) |


### Math
`MultipleNegativesRankingLoss` reduces to the same in-batch InfoNCE used in `src/infonce_inbatch` from SPARQLCE:

```
logits = cos_sim(A, P) * scale
loss   = cross_entropy(logits, arange(N))
```

### Requirements
```
pip install sentence-transformers>=3.0 peft bitsandbytes accelerate transformers datasets
```

In [1]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import json
import random
from pathlib import Path
from collections import defaultdict

import torch
import numpy as np
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    models,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU  : NVIDIA GeForce RTX 3090
VRAM : 25.3 GB


In [11]:
BASE_MODEL          = "Qwen/Qwen3-Embedding-0.6B"
MAX_SEQ_LENGTH      = 512

LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

BATCH_SIZE          = 8
NUM_EPOCHS          = 2
WARMUP_STEPS        = 100
LR                  = 1e-4
SEED                = 42

USE_4BIT            = True

# ── Data paths (must match what SPARQLCE was trained on) ─────────────────
# Point DATA_ROOT at the root of `SPARQL-Embeddings` so we load the SAME
# jaccard triplets + predefined splits that SPARQLCE uses.
DATA_ROOT     = Path(os.environ.get("SPARQL_EMB_ROOT", "../../../SPARQL-Embeddings")).resolve()
DATA_PATH     = DATA_ROOT / "dataset" / "raw"    / "jaccard_triplets_dbpedia.json"
SPLITS_PATH   = DATA_ROOT / "dataset" / "splits" / "splits_dbpedia.json"

OUTPUT_DIR    = Path("..").resolve() / "checkpoints" / "qwen_dbpedia_mnrl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(),   f"Missing triplets file: {DATA_PATH}"
assert SPLITS_PATH.exists(), f"Missing splits   file: {SPLITS_PATH}"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Triplets       : {DATA_PATH}")
print(f"Splits         : {SPLITS_PATH}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Base model     : {BASE_MODEL}")
print(f"LoRA           : r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"Batch / epochs : {BATCH_SIZE} / {NUM_EPOCHS}")
print(f"LR / warmup    : {LR} / {WARMUP_STEPS}")
print(f"4-bit quant    : {USE_4BIT}")

Triplets       : /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/dataset/raw/jaccard_triplets_dbpedia.json
Splits         : /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/dataset/splits/splits_dbpedia.json
Output dir     : /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/baseline/checkpoints/qwen_dbpedia_mnrl
Base model     : Qwen/Qwen3-Embedding-0.6B
LoRA           : r=16, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj']
Batch / epochs : 8 / 2
LR / warmup    : 0.0001 / 100
4-bit quant    : True


In [12]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

with open(SPLITS_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

print(f"Raw triplets   : {len(raw)}")
print(f"Splits file keys:")
for k, v in splits.items():
    if isinstance(v, list):
        print(f"  {k:<12}: list  len={len(v)}  "
              f"sample={v[:3]}{' ...' if len(v) > 3 else ''}")
    elif isinstance(v, dict):
        print(f"  {k:<12}: dict  keys={list(v.keys())}")
    else:
        print(f"  {k:<12}: {type(v).__name__}  value={v}")

def resolve_split(splits_dict, candidates):
    """Return the first list-valued entry whose key matches one of `candidates`."""
    for name in candidates:
        if name in splits_dict and isinstance(splits_dict[name], list):
            return name, splits_dict[name]
    return None, None

TRAIN_KEY, train_idxs = resolve_split(splits, ["train", "training", "train_indices", "train_idx"])
VAL_KEY,   val_idxs   = resolve_split(splits, ["valid", "val", "validation", "valid_indices", "val_indices", "valid_idx", "val_idx"])

assert train_idxs is not None, f"No train list found in splits file. Keys: {list(splits.keys())}"
assert val_idxs   is not None, f"No validation list found in splits file. Keys: {list(splits.keys())}"

print(f"\nUsing train key = '{TRAIN_KEY}'  ({len(train_idxs)} indices)")
print(f"Using val   key = '{VAL_KEY}'  ({len(val_idxs)} indices)")

def get_anchor(item):
    return item.get("anchor_query") or item.get("base_query")

def get_text(q):
    if isinstance(q, dict):
        return q.get("query") or q.get("text")
    return q

def item_to_examples(item):
    """
    Flatten one triplet record into InputExample(anchor, positive, negative).
    Pairs each positive with a random negative from the same record.

    Returns [] if the record has no usable (anchor, pos, neg) combination.
    """
    a = get_anchor(item)
    if not a:
        return []
    positives = [p for p in (get_text(x) for x in item.get("positive_queries") or []) if p and p != a]
    negatives = [n for n in (get_text(x) for x in item.get("negative_queries") or []) if n and n != a]
    if not positives or not negatives:
        return []
    return [InputExample(texts=[a, p, random.choice(negatives)]) for p in positives]

def build_split(name, idxs):
    examples = []
    kept_anchors = 0
    skipped = 0
    for i in idxs:
        if not (0 <= i < len(raw)):
            skipped += 1
            continue
        exs = item_to_examples(raw[i])
        if exs:
            kept_anchors += 1
            examples.extend(exs)
    print(f"  {name:<8}: {len(examples):>6} examples  over {kept_anchors:>5} anchors "
          f"(from {len(idxs)} indices, skipped_oor={skipped})")
    return examples

print("\nBuilding split-aligned InputExamples (anchor, positive, negative):")
train_examples = build_split(TRAIN_KEY, train_idxs)
val_examples   = build_split(VAL_KEY,   val_idxs)

train_anchors = {ex.texts[0] for ex in train_examples}
val_anchors   = {ex.texts[0] for ex in val_examples}
overlap       = train_anchors & val_anchors
print(f"\nAnchor overlap train∩val: {len(overlap)}  "
      f"({'OK' if not overlap else 'LEAKAGE — check splits file'})")

Raw triplets   : 4713
Splits file keys:
  seed        : int  value=42
  train_idx   : list  len=3770  sample=[283, 4695, 256] ...
  valid_idx   : list  len=471  sample=[4572, 3378, 2107] ...
  test_idx    : list  len=472  sample=[411, 3805, 1440] ...

Using train key = 'train_idx'  (3770 indices)
Using val   key = 'valid_idx'  (471 indices)

Building split-aligned InputExamples (anchor, positive, negative):
  train_idx:  31477 examples  over  3770 anchors (from 3770 indices, skipped_oor=0)
  valid_idx:   4097 examples  over   471 anchors (from 471 indices, skipped_oor=0)

Anchor overlap train∩val: 0  (OK)


In [13]:
print(f"Train examples : {len(train_examples)}")
print(f"Val   examples : {len(val_examples)}")
print("(Splits are loaded from splits_dbpedia.json — same partitioning as SPARQLCE.)")

Train examples : 31477
Val   examples : 4097
(Splits are loaded from splits_dbpedia.json — same partitioning as SPARQLCE.)


In [14]:
def build_ir_evaluator(val_exs, name="dbpedia_val_ir"):
    """
    Build an InformationRetrievalEvaluator that measures MRR@k / nDCG@k /
    Recall@k during training. This is directly aligned with the retrieval
    task you evaluate with in `eval/07_*`, unlike pair-accuracy.

    corpus  : union of all positives and negatives across val triples
    queries : unique anchors in val
    qrels   : for each anchor, the set of its positives
    """
    corpus    = {}    # doc_id -> text
    queries   = {}    # query_id -> anchor text
    qrels     = defaultdict(set)    # query_id -> set(doc_id)
    text_to_doc = {}

    def add_doc(text):
        if text not in text_to_doc:
            did = f"d{len(text_to_doc)}"
            text_to_doc[text] = did
            corpus[did] = text
        return text_to_doc[text]

    anchor_to_qid = {}
    for ex in val_exs:
        a, p, n = ex.texts
        if a not in anchor_to_qid:
            qid = f"q{len(anchor_to_qid)}"
            anchor_to_qid[a] = qid
            queries[qid] = a
        qid = anchor_to_qid[a]
        p_did = add_doc(p)
        _     = add_doc(n)
        qrels[qid].add(p_did)

    qrels = {q: set(ds) for q, ds in qrels.items()}

    print(f"IR evaluator: {len(queries)} queries / {len(corpus)} docs / "
          f"{sum(len(v) for v in qrels.values())} qrels")

    return InformationRetrievalEvaluator(
        queries           = queries,
        corpus            = corpus,
        relevant_docs     = qrels,
        mrr_at_k          = [10],
        ndcg_at_k         = [10],
        accuracy_at_k     = [1, 3, 5, 10],
        precision_recall_at_k = [3, 5, 10],
        name              = name,
        show_progress_bar = False,
        batch_size        = 32,
    )

ir_evaluator = build_ir_evaluator(val_examples)

IR evaluator: 471 queries / 1556 docs / 4097 qrels


In [15]:
def build_qwen_st_with_lora(base_model_name, use_4bit=True, max_seq_length=512):
    """
    Load Qwen3-Embedding once, wrap with LoRA, expose as a SentenceTransformer
    with last-token pooling.
    """
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_name, trust_remote_code=True, padding_side="left"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = None
    if use_4bit and torch.cuda.is_available():
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_quant_type       = "nf4",
            bnb_4bit_compute_dtype    = torch.bfloat16,
            bnb_4bit_use_double_quant = True,
        )

    base_model = AutoModel.from_pretrained(
        base_model_name,
        quantization_config = bnb_config,
        torch_dtype         = torch.float16 if (torch.cuda.is_available() and not use_4bit) else None,
        trust_remote_code   = True,
        device_map          = None,
    )

    lora_cfg = LoraConfig(
        r              = LORA_R,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        target_modules = LORA_TARGET_MODULES,
        bias           = "none",
        task_type      = TaskType.FEATURE_EXTRACTION,
    )
    peft_base = get_peft_model(base_model, lora_cfg)
    peft_base.print_trainable_parameters()

    word_module = models.Transformer(
        base_model_name,
        max_seq_length = max_seq_length,
        tokenizer_args = {"trust_remote_code": True, "padding_side": "left"},
        model_args     = {"trust_remote_code": True},
    )
    word_module.auto_model = peft_base

    pooling = models.Pooling(
        word_module.get_word_embedding_dimension(),
        pooling_mode_lasttoken    = True,
        pooling_mode_mean_tokens  = False,
    )

    return SentenceTransformer(modules=[word_module, pooling], device=device)

model = build_qwen_st_with_lora(BASE_MODEL, use_4bit=USE_4BIT, max_seq_length=MAX_SEQ_LENGTH)

Loading weights:   0%|          | 1/310 [00:00<04:05,  1.26it/s]/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 310/310 [00:01<00:00, 298.64it/s]


trainable params: 4,587,520 || all params: 600,364,032 || trainable%: 0.7641


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4487.11it/s]


In [16]:
train_loader = DataLoader(
    train_examples,
    shuffle    = True,
    batch_size = BATCH_SIZE,
    drop_last  = True,
)

train_loss = losses.MultipleNegativesRankingLoss(
    model,
    scale = 20.0,
)

steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * NUM_EPOCHS
print(f"Steps / epoch : {steps_per_epoch}")
print(f"Total steps   : {total_steps}")
print(f"Warmup steps  : {WARMUP_STEPS}")

Steps / epoch : 3934
Total steps   : 7868
Warmup steps  : 100


In [17]:
model.fit(
    train_objectives     = [(train_loader, train_loss)],
    evaluator            = ir_evaluator,
    epochs               = NUM_EPOCHS,
    warmup_steps         = WARMUP_STEPS,
    optimizer_params     = {"lr": LR},
    output_path          = str(OUTPUT_DIR),
    save_best_model      = True,
    evaluation_steps     = steps_per_epoch,
    show_progress_bar    = True,
    use_amp              = False,
    checkpoint_path      = str(OUTPUT_DIR / "checkpoints"),
    checkpoint_save_steps= steps_per_epoch,
)

print("\n✅ Training done.")

Step,Training Loss,Validation Loss,Dbpedia Val Ir Cosine Accuracy@1,Dbpedia Val Ir Cosine Accuracy@3,Dbpedia Val Ir Cosine Accuracy@5,Dbpedia Val Ir Cosine Accuracy@10,Dbpedia Val Ir Cosine Precision@3,Dbpedia Val Ir Cosine Precision@5,Dbpedia Val Ir Cosine Precision@10,Dbpedia Val Ir Cosine Recall@3,Dbpedia Val Ir Cosine Recall@5,Dbpedia Val Ir Cosine Recall@10,Dbpedia Val Ir Cosine Ndcg@10,Dbpedia Val Ir Cosine Mrr@10,Dbpedia Val Ir Cosine Map@100
3934,0.342820,No log,0.358811,0.679406,0.743100,0.802548,0.338995,0.296815,0.248408,0.396406,0.442022,0.529566,0.535215,0.530693,0.527318
7868,0.242008,No log,0.373673,0.694268,0.743100,0.813163,0.347488,0.311677,0.273248,0.397013,0.448160,0.555331,0.561502,0.543002,0.559919



✅ Training done.


In [18]:
from peft import PeftModel

final_merged_dir = OUTPUT_DIR / "final_merged"
final_merged_dir.mkdir(parents=True, exist_ok=True)

word_module = model[0]
inner       = word_module.auto_model

def find_peft(module):
    if isinstance(module, PeftModel):
        return module
    for child in module.modules():
        if isinstance(child, PeftModel):
            return child
    return None

peft_model = find_peft(inner)
if peft_model is not None:
    print("Merging LoRA weights into base...")
    word_module.auto_model = peft_model.merge_and_unload()
else:
    print("No PeftModel wrapper detected; saving current weights as-is.")

model.save(str(final_merged_dir))
print(f"✅ Saved merged model → {final_merged_dir}")

Merging LoRA weights into base...


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]


✅ Saved merged model → /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/baseline/checkpoints/qwen_dbpedia_mnrl/final_merged


In [19]:
print("Reloading merged model to verify clean load...")
reloaded = SentenceTransformer(str(final_merged_dir))
scores = ir_evaluator(reloaded)

print("\nIR metrics on val (reloaded):")
for k, v in scores.items():
    print(f"  {k}: {v:.4f}")

del reloaded
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Reloading merged model to verify clean load...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 975.54it/s] 



IR metrics on val (reloaded):
  dbpedia_val_ir_cosine_accuracy@1: 0.2569
  dbpedia_val_ir_cosine_accuracy@3: 0.4522
  dbpedia_val_ir_cosine_accuracy@5: 0.5138
  dbpedia_val_ir_cosine_accuracy@10: 0.5966
  dbpedia_val_ir_cosine_precision@3: 0.2074
  dbpedia_val_ir_cosine_precision@5: 0.1703
  dbpedia_val_ir_cosine_precision@10: 0.1195
  dbpedia_val_ir_cosine_recall@3: 0.2718
  dbpedia_val_ir_cosine_recall@5: 0.3054
  dbpedia_val_ir_cosine_recall@10: 0.3445
  dbpedia_val_ir_cosine_ndcg@10: 0.3343
  dbpedia_val_ir_cosine_mrr@10: 0.3693
  dbpedia_val_ir_cosine_map@100: 0.3131


### Next step

Evaluate on the full test-anchor pool by running `eval/08_eval_qwen_mnrl_dbpedia.ipynb` (created alongside this notebook). It points at `checkpoints/qwen_dbpedia_mnrl/final_merged` and reuses the same `evaluate_sentence_transformer` helper as the other eval notebooks so results are directly comparable to base Qwen, the old fine-tuned Qwen, and SPARQLCE.